In [2]:
import torch
from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    Trainer,
    TrainingArguments
)
from datasets import load_from_disk
import numpy as np

c:\Users\win11\anaconda3\envs\nlp-haoussa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "google/mt5-small"

TRAIN_PATH = "../data/processed/train_dataset"
VAL_PATH = "../data/processed/val_dataset"

OUTPUT_DIR = "../models/mt5-haoussa-zarma"

In [4]:
train_dataset = load_from_disk(TRAIN_PATH)
val_dataset = load_from_disk(VAL_PATH)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 1063
Val size: 133


In [7]:
model = MT5ForConditionalGeneration.from_pretrained(MODEL_NAME, weights_only=False)
tokenizer = MT5Tokenizer.from_pretrained(MODEL_NAME)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [8]:
print(torch.__version__)

2.5.1+cu121


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model.to(device)

Device: cuda


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [10]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [11]:
import evaluate

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    result = bleu.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": result["score"]}

In [16]:
from transformers import Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    # --- Chemins et Sauvegarde ---
    output_dir=OUTPUT_DIR,
    logging_dir="../logs",
    save_total_limit=2,             # Garde seulement les 2 meilleurs modèles
    save_strategy="epoch",
    
    # --- Stratégie d'Entraînement ---
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16, # Augmenté pour saturer ton GPU
    num_train_epochs=15,
    weight_decay=0.01,
    
    # --- Accélération Hardware (NVIDIA) ---
    fp16=True,                      # Utilise les coeurs Tensor (indispensable avec CUDA)
    dataloader_num_workers=2,       # Utilise le CPU pour préparer les données en parallèle
    
    # --- Paramètres Spécifiques Seq2Seq (Génération) ---
    predict_with_generate=True,     # Permet de calculer le BLEU/ROUGE
    generation_max_length=64,       # Évite les phrases trop longues qui saturent la RAM
    per_device_eval_batch_size=4,   # Augmenté légèrement car fp16 consomme moins
    
    # --- Gestion de la Mémoire (Anti-Crash) ---
    eval_accumulation_steps=10,     # Décharge les prédictions tous les 10 batches
    
    # --- Monitoring ---
    logging_steps=50,
    report_to="none",               # Mets "tensorboard" si tu veux des graphiques plus tard
)

In [17]:
import torch
import gc

# Juste avant trainer.train()
gc.collect()
torch.cuda.empty_cache()

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

C:\Users\win11\AppData\Local\Temp\ipykernel_11820\3740418003.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
def translate(text):
    input_text = "translate Hausa to Zarma: " + text

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = model.generate(**inputs, max_length=32)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(translate("Sannu"))
print(translate("Ina gidan ku?"))

In [ ]:
"""
Observations:

1. Le modèle converge progressivement
2. BLEU score utilisé pour évaluer qualité traduction
3. Dataset limité → apprentissage rapide mais risque d’overfitting
4. mT5 adapté aux langues multiples mais nécessite fine-tuning

Conclusion:
Le modèle apprend des correspondances basiques Hausa → Zarma.
"""